# 2.1 - Data Wrangling for GenAI

**Module 2 - Data Prerequisites** | Netsetos GenAI Engineering

This notebook runs the one piece of the lesson that actually executes end to end: the defensive text-cleaning pipeline. It first forges a deliberately broken text file (BOM, zero-width characters, non-breaking spaces, whitespace runs), then applies the five named defenses every production text pipeline needs before data touches an embedding model or a fine-tuning API.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Installs the data-wrangling stack referenced across the lesson and pins the imports the runnable demo needs. Only `unicodedata`, `re`, and `tiktoken` are actually exercised below; the rest (Polars, pandas, sklearn, langchain, pydantic, unstructured) back the reference snippets in the markdown.

In [ ]:
# Install dependencies (if running in Colab or fresh environment)
!pip install -q pandas scikit-learn langchain langchain-text-splitters polars tiktoken unstructured pydantic

# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

A one-time environment prep cell - it pip-installs the full toolkit and leaves a note that the lesson uses seeded randomness. Nothing here runs a pipeline; it just makes the later cells importable.

**How the code works - step by step**
- **`pip install ...`** - pulls the wrangling stack: `pandas` + `scikit-learn` (interop and splits), `langchain` + `langchain-text-splitters` (chunking), `polars` (the 2026 default DataFrame), `tiktoken` (OpenAI tokenizer for token-aware truncation), `unstructured` (layout-aware PDF parsing), `pydantic` (schema validation).
- **reproducibility comment** - flags that seeded random values are used later so splits and samples are repeatable.

**In one line:** install everything the lesson references, even though only the text-defense trio runs here.

**What you'll see:** (no output - environment prep)

## 1 - Forge a deliberately hostile text file

Before you can defend against messy text you need messy text. This cell writes a small `messy.txt` on disk that packs in the four corruptions that bite real pipelines - a UTF-8 BOM at the very start, zero-width characters hidden between words, non-breaking spaces masquerading as normal spaces, and ragged runs of whitespace. It exists so the next cell has something real to clean on a Restart & Run All.

In [ ]:
# Setup: write a small messy.txt so the text-defense demo below runs on Restart & Run All.
# It intentionally contains a UTF-8 BOM, zero-width chars, non-breaking spaces, and whitespace runs.
bom, zwsp, zwnj, nbsp, nl = chr(0xFEFF), chr(0x200B), chr(0x200C), chr(0xA0), chr(10)
messy = bom + '  Hello' + zwsp + '  world' + zwnj + nl + nl + '  cafe' + nbsp + nbsp + 'test    done  '
with open('messy.txt', 'w', encoding='utf-8') as _f:
    _f.write(messy)


**Explanation**

This is a fixture-writer, not part of the pipeline - it manufactures the exact broken input the demo below is designed to survive. Each character it injects (BOM, ZWSP, ZWNJ, NBSP) is chosen to trip a naive `str.strip()`.

**How the code works - step by step**
- **`bom, zwsp, zwnj, nbsp, nl = chr(...)`** - names the troublemakers by codepoint: `U+FEFF` byte-order mark, `U+200B` zero-width space, `U+200C` zero-width non-joiner, `U+00A0` non-breaking space, and a newline.
- **`messy = bom + ...`** - concatenates them into a string that leads with a BOM, hides zero-width chars inside `Hello world`, doubles up newlines, and glues `cafe` to `test` with non-breaking spaces plus trailing whitespace.
- **`open(..., 'w', encoding='utf-8')`** - writes the string to `messy.txt` so it lives on disk for the next cell to read.

**In one line:** hand-build a file containing every corruption the defense below is meant to strip.

**What you'll see:** No printed output - it silently creates a small file named `messy.txt` in the working directory.

## 2 - Clean real-world text with five named defenses

This is the core runnable demo of the lesson. It reads the hostile `messy.txt` and applies five ordered transforms - BOM-aware decoding, Unicode normalization, zero-width stripping, whitespace collapse, and tokenizer-aware truncation - each fixing one specific failure mode. The order matters: you decode safely first, normalize form, then remove and collapse, and only cut at a real token boundary last.

In [ ]:
import unicodedata, re, tiktoken

# 1. Always read with explicit encoding + replace policy
with open('messy.txt', encoding='utf-8-sig', errors='replace') as f:
    text = f.read()
# utf-8-sig strips BOM; errors=replace substitutes bad bytes.

# 2. Normalize Unicode form (combining characters)
text = unicodedata.normalize('NFKC', text)

# 3. Strip zero-width characters (U+200B-200D, U+FEFF)
text = re.sub(r'[​-‏‪-‮⁠﻿]', '', text)

# 4. Collapse whitespace runs
text = re.sub(r'\s+', ' ', text).strip()

# 5. Tokenizer-aware truncation
enc = tiktoken.encoding_for_model('gpt-4o')
tokens = enc.encode(text)
text_8k = enc.decode(tokens[:8000])
# Output:
# 5 defensive transforms applied. Text is now safe for embedding / fine-tuning.

**Explanation**

A defensive-text toolkit run as a straight-line script, not a model call - it takes dirty bytes on disk and returns a string that is safe to embed or fine-tune on. Read it as five independent guards stacked in the order they must fire.

**How the code works - step by step**
- **`open(..., encoding='utf-8-sig', errors='replace')`** - reads with an explicit encoding: `utf-8-sig` strips the leading BOM, and `errors='replace'` swaps any undecodable byte for a placeholder instead of crashing mid-file.
- **`unicodedata.normalize('NFKC', text)`** - folds combining characters and compatibility forms into one canonical representation so visually identical strings compare equal.
- **`re.sub(r'[...zero-width...]', '', text)`** - deletes the invisible zero-width characters (`U+200B`-`U+200D`, `U+FEFF`) that survive a normal strip.
- **`re.sub(r'\s+', ' ', text).strip()`** - collapses every run of whitespace (including the non-breaking spaces and doubled newlines) into single spaces and trims the ends.
- **`enc = tiktoken.encoding_for_model('gpt-4o')` -> `enc.encode` / `enc.decode(tokens[:8000])`** - truncates by *token* boundary, not character count, so nothing is ever cut mid-multibyte-sequence.

**In one line:** decode safely -> normalize -> strip invisibles -> collapse whitespace -> cut at a token boundary.

**What you'll see:** No visible print - the `# Output:` comment states the takeaway: 5 defensive transforms are applied and the text is now safe for embedding or fine-tuning. Inspect `text` or `tokens` yourself to confirm the BOM and zero-width characters are gone.

Everything in this notebook orbits one runnable lesson: real-world text is hostile, and you defend against it explicitly. You built a file carrying every common corruption - a BOM, zero-width joiners, non-breaking spaces, ragged whitespace - then neutralized it with five ordered transforms and cut it at a real token boundary instead of a character count. The format, Polars-vs-Pandas, PDF-ingestion, and schema-validation topics live in the surrounding markdown as reference snippets; the token-safe cleaning you ran here is the primitive they all depend on. Next, Lesson 2.2 (SQL for GenAI) queries the Parquet and JSONL these pipelines produce, and Module 8.2 expands the parse-filter-chunk pipeline into a full RAG ingestion system.